In [ ]:
import json
import csv
import re
import random
from pathlib import Path
from typing import Any, Dict, List, Optional, Iterable, Tuple


# ============================================================
# CONFIG
# ============================================================

ROOT = Path(r"D:\Users\TimeBound")

OUTPUT_DIR = ROOT / "converted_external"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42
random.seed(RANDOM_SEED)

DEFAULT_CURRENT_TIME = "2026-01-01T12:00:00"

DATASET_ALIASES = {
    "tempreason": [
        "TempReason",
        "tempreason",
        "temp_reason",
        "Temp Reason",
    ],
    "complextr": [
        "complex4r",
        "complex-tr",
        "Complex-TR",
        "complex_tr",
        "ComplexTR",
        "complextr",
    ],
    "tcp": [
        "TCP",
        "tcp",
        "Temporal Constraint",
        "TemporalConstraint",
    ],
}

SUPPORTED_EXTS = {
    ".json",
    ".jsonl",
    ".ndjson",
    ".csv",
    ".tsv",
}


# ============================================================
# Generic helpers
# ============================================================

def norm_key(s: str) -> str:
    return str(s).strip().lower().replace("-", "_").replace(" ", "_")


def safe_str(x: Any) -> str:
    if x is None:
        return ""
    if isinstance(x, str):
        return x
    if isinstance(x, (int, float, bool)):
        return str(x)
    try:
        return json.dumps(x, ensure_ascii=False)
    except Exception:
        return str(x)


def compact_text(s: str) -> str:
    s = safe_str(s)
    s = re.sub(r"\s+", " ", s).strip()
    return s


def find_first_key(obj: Dict[str, Any], keys: List[str]) -> Optional[Any]:
    if not isinstance(obj, dict):
        return None

    key_map = {norm_key(k): k for k in obj.keys()}

    for k in keys:
        nk = norm_key(k)
        if nk in key_map:
            return obj[key_map[nk]]

    return None


def has_any_key(obj: Dict[str, Any], keys: List[str]) -> bool:
    if not isinstance(obj, dict):
        return False

    obj_keys = {norm_key(k) for k in obj.keys()}
    return any(norm_key(k) in obj_keys for k in keys)


def maybe_parse_json_string(x: Any) -> Any:
    if not isinstance(x, str):
        return x

    s = x.strip()
    if not s:
        return x

    if not ((s.startswith("{") and s.endswith("}")) or (s.startswith("[") and s.endswith("]"))):
        return x

    try:
        return json.loads(s)
    except Exception:
        return x


def split_sentences(text: str) -> List[str]:
    text = compact_text(text)

    if not text:
        return []

    # Try split on sentence boundaries.
    parts = re.split(r"(?<=[.!?])\s+", text)
    parts = [p.strip() for p in parts if p.strip()]

    # If one huge paragraph, chunk it.
    if len(parts) <= 1 and len(text) > 500:
        chunks = []
        words = text.split()
        step = 60

        for i in range(0, len(words), step):
            chunk = " ".join(words[i:i + step]).strip()
            if chunk:
                chunks.append(chunk)

        return chunks

    return parts


def is_timestamp_like(x: Any) -> bool:
    s = safe_str(x)
    return bool(
        re.search(r"\d{4}-\d{2}-\d{2}", s)
        or re.search(r"\b\d{1,2}:\d{2}\b", s)
        or re.search(r"\b(Monday|Tuesday|Wednesday|Thursday|Friday|Saturday|Sunday)\b", s, re.I)
        or re.search(r"\b(today|tomorrow|yesterday|next|before|after|during|until|since)\b", s, re.I)
    )


def difficulty_from_history(n_turns: int) -> str:
    if n_turns <= 3:
        return "easy"
    if n_turns <= 8:
        return "medium"
    return "hard"


def normalize_answer(ans: Any) -> str:
    ans = maybe_parse_json_string(ans)

    if ans is None:
        return ""

    if isinstance(ans, list):
        if not ans:
            return ""
        # If list of strings, join short answers.
        if all(isinstance(x, (str, int, float, bool)) for x in ans):
            return "; ".join(safe_str(x) for x in ans)
        return json.dumps(ans, ensure_ascii=False)

    if isinstance(ans, dict):
        for k in [
            "answer",
            "gold_answer",
            "label",
            "target",
            "text",
            "value",
            "final_answer",
            "output",
        ]:
            v = find_first_key(ans, [k])
            if v is not None:
                return normalize_answer(v)
        return json.dumps(ans, ensure_ascii=False)

    return compact_text(ans)


# ============================================================
# Folder discovery
# ============================================================

def find_dataset_folder(root: Path, aliases: List[str]) -> Optional[Path]:
    if not root.exists():
        raise FileNotFoundError(f"Root does not exist: {root}")

    candidates = []

    for p in root.iterdir():
        if not p.is_dir():
            continue

        name = p.name.lower()

        for alias in aliases:
            a = alias.lower()
            if a in name or name in a:
                candidates.append(p)

    if candidates:
        candidates = sorted(candidates, key=lambda x: len(str(x)))
        return candidates[0]

    # recursive fallback, shallow enough
    for p in root.rglob("*"):
        if not p.is_dir():
            continue

        name = p.name.lower()

        for alias in aliases:
            a = alias.lower()
            if a in name or name in a:
                return p

    return None


def list_supported_files(folder: Path) -> List[Path]:
    files = []

    for p in folder.rglob("*"):
        if p.is_file() and p.suffix.lower() in SUPPORTED_EXTS:
            # Skip common metadata files that usually do not contain examples.
            low = p.name.lower()
            if low in {"dataset_infos.json", "state.json"}:
                continue
            if low.startswith("."):
                continue
            files.append(p)

    return sorted(files)


# ============================================================
# File loading
# ============================================================

def load_json_file(path: Path) -> Any:
    text = path.read_text(encoding="utf-8", errors="ignore").strip()

    if not text:
        return []

    try:
        return json.loads(text)
    except Exception:
        # Sometimes .json contains jsonl.
        rows = []
        for line in text.splitlines():
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except Exception:
                pass
        return rows


def load_jsonl_file(path: Path) -> List[Any]:
    rows = []

    with path.open("r", encoding="utf-8", errors="ignore") as f:
        for line_no, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue

            try:
                rows.append(json.loads(line))
            except Exception:
                # Keep raw line as text record if needed.
                rows.append({"text": line, "_line_no": line_no})

    return rows


def load_csv_or_tsv(path: Path) -> List[Dict[str, Any]]:
    rows = []
    delimiter = "\t" if path.suffix.lower() == ".tsv" else ","

    with path.open("r", encoding="utf-8", errors="ignore", newline="") as f:
        reader = csv.DictReader(f, delimiter=delimiter)

        for row in reader:
            rows.append(dict(row))

    return rows


def load_file(path: Path) -> List[Any]:
    ext = path.suffix.lower()

    if ext == ".json":
        obj = load_json_file(path)
        if isinstance(obj, list):
            return obj
        return [obj]

    if ext in {".jsonl", ".ndjson"}:
        return load_jsonl_file(path)

    if ext in {".csv", ".tsv"}:
        return load_csv_or_tsv(path)

    return []


# ============================================================
# Record detection and extraction
# ============================================================

QUESTION_KEYS = [
    "question",
    "query",
    "input",
    "prompt",
    "q",
    "final_query",
    "user_query",
    "task",
    "instruction",
]

ANSWER_KEYS = [
    "answer",
    "answers",
    "gold_answer",
    "label",
    "target",
    "output",
    "outputs",
    "completion",
    "response",
    "final_answer",
    "correct_answer",
]

CONTEXT_KEYS = [
    "context",
    "passage",
    "story",
    "text",
    "background",
    "facts",
    "evidence",
    "paragraph",
    "document",
    "documents",
    "premise",
    "source",
    "source_text",
    "constraints",
    "constraint",
]

DIALOGUE_KEYS = [
    "dialogue",
    "dialog",
    "conversation",
    "messages",
    "utterances",
    "turns",
    "history",
]

TIME_KEYS = [
    "current_time",
    "timestamp",
    "time",
    "date",
    "reference_time",
    "query_time",
    "now",
]

SUPPORT_KEYS = [
    "evidence_turns",
    "gold_evidence_turns",
    "supporting_facts",
    "supports",
    "support",
    "evidence_indices",
    "evidence_ids",
]


def looks_like_record(obj: Any) -> bool:
    if not isinstance(obj, dict):
        return False

    has_q = has_any_key(obj, QUESTION_KEYS)
    has_a = has_any_key(obj, ANSWER_KEYS)
    has_ctx = has_any_key(obj, CONTEXT_KEYS) or has_any_key(obj, DIALOGUE_KEYS)

    # QA-style
    if has_q and has_a:
        return True

    # Dialogue/task style
    if has_ctx and has_a:
        return True

    # Some TCP-style records may have prompt/completion.
    if has_any_key(obj, ["prompt"]) and has_any_key(obj, ["completion"]):
        return True

    return False


def collect_records_from_obj(obj: Any, source_file: str, split_hint: str = "") -> List[Dict[str, Any]]:
    """
    Recursively collect dicts that look like dataset examples.
    Handles:
      - list of examples
      - dict with train/dev/test lists
      - nested HuggingFace-like JSON structures
    """
    records = []

    def rec(x: Any, path: str):
        x = maybe_parse_json_string(x)

        if isinstance(x, dict):
            if looks_like_record(x):
                y = dict(x)
                y["_source_file"] = source_file
                y["_source_path"] = path
                if split_hint:
                    y["_split_hint"] = split_hint
                records.append(y)
                return

            # common split containers
            for k, v in x.items():
                nk = norm_key(k)
                if nk in {"train", "validation", "valid", "dev", "test"}:
                    rec(v, f"{path}/{k}")
                else:
                    rec(v, f"{path}/{k}")

        elif isinstance(x, list):
            for i, item in enumerate(x):
                rec(item, f"{path}[{i}]")

    rec(obj, "$")
    return records


def load_dataset_records(folder: Path) -> List[Dict[str, Any]]:
    files = list_supported_files(folder)

    print(f"Found {len(files)} supported files in {folder}")

    all_records = []

    for path in files:
        try:
            raw = load_file(path)
            records = collect_records_from_obj(raw, source_file=str(path))
            if records:
                print(f"  {path.relative_to(folder)} -> {len(records)} records")
            all_records.extend(records)
        except Exception as e:
            print(f"  SKIP {path}: {e}")

    return all_records


# ============================================================
# Context/history extraction
# ============================================================

def extract_question(record: Dict[str, Any]) -> str:
    v = find_first_key(record, QUESTION_KEYS)

    if v is None:
        # fallback for prompt/completion
        v = find_first_key(record, ["prompt"])

    return compact_text(v)


def extract_answer(record: Dict[str, Any]) -> str:
    v = find_first_key(record, ANSWER_KEYS)

    if v is None:
        return ""

    return normalize_answer(v)


def extract_current_time(record: Dict[str, Any]) -> str:
    v = find_first_key(record, TIME_KEYS)

    if v is not None and is_timestamp_like(v):
        s = compact_text(v)
        # Do not over-parse; keep as given if it contains date-ish content.
        if re.search(r"\d{4}-\d{2}-\d{2}", s):
            if "T" in s:
                return s
            return s + "T12:00:00"

    # Try finding date in question.
    q = extract_question(record)
    m = re.search(r"\d{4}-\d{2}-\d{2}", q)
    if m:
        return m.group(0) + "T12:00:00"

    return DEFAULT_CURRENT_TIME


def text_from_turn_obj(turn: Any) -> Tuple[str, str]:
    """
    Returns speaker, text.
    """
    if isinstance(turn, str):
        return "unknown", compact_text(turn)

    if isinstance(turn, dict):
        speaker = find_first_key(turn, ["speaker", "role", "from", "agent", "user", "name"])
        if isinstance(speaker, bool):
            speaker = "user" if speaker else "assistant"

        text = find_first_key(
            turn,
            [
                "text",
                "content",
                "utterance",
                "message",
                "value",
                "sentence",
                "turn",
                "dialogue_act",
            ],
        )

        if text is None:
            # fallback: compact dict
            text = json.dumps(turn, ensure_ascii=False)

        return compact_text(speaker or "unknown"), compact_text(text)

    return "unknown", compact_text(turn)


def extract_dialogue_turns(record: Dict[str, Any]) -> List[str]:
    v = find_first_key(record, DIALOGUE_KEYS)
    v = maybe_parse_json_string(v)

    if v is None:
        return []

    turns = []

    if isinstance(v, list):
        for item in v:
            speaker, text = text_from_turn_obj(item)
            if text:
                if speaker and speaker != "unknown":
                    turns.append(f"{speaker}: {text}")
                else:
                    turns.append(text)

    elif isinstance(v, dict):
        # Sometimes dialogue is dict of turns.
        for k, item in v.items():
            speaker, text = text_from_turn_obj(item)
            if text:
                turns.append(f"{speaker}: {text}" if speaker != "unknown" else text)

    elif isinstance(v, str):
        # Split by line if possible.
        lines = [compact_text(x) for x in v.splitlines() if compact_text(x)]
        if len(lines) >= 2:
            turns.extend(lines)
        else:
            turns.extend(split_sentences(v))

    return [x for x in turns if x]


def extract_context_turns(record: Dict[str, Any]) -> List[str]:
    # Prefer dialogue if present.
    dialogue = extract_dialogue_turns(record)
    if dialogue:
        return dialogue

    pieces = []

    for key in CONTEXT_KEYS:
        v = find_first_key(record, [key])
        v = maybe_parse_json_string(v)

        if v is None:
            continue

        if isinstance(v, str):
            pieces.extend(split_sentences(v))

        elif isinstance(v, list):
            for item in v:
                item = maybe_parse_json_string(item)

                if isinstance(item, str):
                    pieces.append(compact_text(item))
                elif isinstance(item, dict):
                    speaker, text = text_from_turn_obj(item)
                    if text:
                        pieces.append(text)

        elif isinstance(v, dict):
            # If dict with values as strings/list, flatten lightly.
            for kk, vv in v.items():
                vv = maybe_parse_json_string(vv)
                if isinstance(vv, str):
                    pieces.extend(split_sentences(vv))
                elif isinstance(vv, list):
                    for item in vv:
                        if isinstance(item, str):
                            pieces.append(compact_text(item))
                        elif isinstance(item, dict):
                            _, text = text_from_turn_obj(item)
                            if text:
                                pieces.append(text)

    # Remove duplicates while preserving order.
    seen = set()
    out = []
    for p in pieces:
        p = compact_text(p)
        if not p:
            continue
        if p.lower() in seen:
            continue
        seen.add(p.lower())
        out.append(p)

    return out


def extract_support_turns(record: Dict[str, Any], n_turns: int) -> Optional[List[int]]:
    v = find_first_key(record, SUPPORT_KEYS)
    v = maybe_parse_json_string(v)

    if v is None:
        return None

    indices = []

    def add_idx(x):
        if isinstance(x, int):
            indices.append(x)
        elif isinstance(x, str):
            # collect integers from string
            for m in re.findall(r"\d+", x):
                indices.append(int(m))
        elif isinstance(x, list):
            for y in x:
                add_idx(y)
        elif isinstance(x, dict):
            for y in x.values():
                add_idx(y)

    add_idx(v)

    indices = [i for i in indices if 0 <= i < n_turns]
    indices = sorted(set(indices))

    if not indices:
        return None

    return indices


def infer_task_type(dataset_name: str, record: Dict[str, Any], question: str, context_turns: List[str]) -> str:
    q = question.lower()
    all_text = " ".join([question] + context_turns).lower()

    if "schedule" in q or "meeting" in q or "appointment" in q or "booking" in q:
        return "rescheduling"

    if "cancel" in all_text or "cancelled" in all_text or "canceled" in all_text:
        return "cancellation"

    if "duration" in q or "how long" in q or "elapsed" in q or "how many days" in q or "how many hours" in q:
        return "elapsed_time_reasoning"

    if "before" in q or "after" in q or "during" in q or "when" in q:
        return "time_window_retrieval"

    if "every" in all_text or "weekly" in all_text or "monthly" in all_text or "daily" in all_text:
        return "periodic_event"

    if dataset_name == "tcp":
        return "time_window_retrieval"

    if dataset_name in {"tempreason", "complextr"}:
        return "time_window_retrieval"

    return "time_window_retrieval"


def make_history(
    context_turns: List[str],
    current_time: str,
) -> List[Dict[str, Any]]:
    history = []

    # External datasets often do not provide event_time / validity interval.
    # We keep temporal metadata explicit but mostly unknown.
    for i, text in enumerate(context_turns):
        history.append(
            {
                "turn": i,
                "speaker": "source",
                "observation_time": current_time,
                "event_time": None,
                "valid_from": None,
                "valid_to": None,
                "status": "historical",
                "text": text,
            }
        )

    return history


def convert_record_to_timebound(
    record: Dict[str, Any],
    dataset_name: str,
    index: int,
) -> Optional[Dict[str, Any]]:
    question = extract_question(record)
    answer = extract_answer(record)
    current_time = extract_current_time(record)

    if not question or not answer:
        return None

    context_turns = extract_context_turns(record)

    # If no context exists, use question itself as minimal context.
    # This keeps wrapper runnable, but retrieval metrics are not meaningful.
    if not context_turns:
        context_turns = [question]

    # Keep context length controlled for external QA.
    # For RAG comparison, full history can still be bigger if source provides it.
    context_turns = [compact_text(x) for x in context_turns if compact_text(x)]

    if not context_turns:
        return None

    history = make_history(context_turns, current_time)
    support_turns = extract_support_turns(record, len(history))

    if support_turns is None:
        # External datasets often do not provide gold evidence.
        # For oracle_evidence we need some evidence; use all turns.
        # Marked in metadata below.
        support_turns = list(range(len(history)))
        evidence_is_heuristic = True
    else:
        evidence_is_heuristic = False

    task_type = infer_task_type(dataset_name, record, question, context_turns)

    ex = {
        "id": f"{dataset_name}_{index:06d}",
        "task_type": task_type,
        "difficulty": difficulty_from_history(len(history)),
        "current_time": current_time,
        "history": history,
        "query": question,
        "gold_answer": answer,
        "gold_evidence_turns": support_turns,
        "required_temporal_operation": "external temporal reasoning / QA wrapper",
        "temporal_trap": "External dataset instance; temporal evidence may be implicit in context.",
        "explanation": "Converted from external dataset into TimeBound-compatible format.",
        "metadata": {
            "source_dataset": dataset_name,
            "source_file": record.get("_source_file", ""),
            "source_path": record.get("_source_path", ""),
            "split_hint": record.get("_split_hint", ""),
            "evidence_is_heuristic": evidence_is_heuristic,
        },
    }

    return ex


# ============================================================
# Conversion pipeline
# ============================================================

def deduplicate_examples(examples: List[Dict[str, Any]], dataset_name: str) -> List[Dict[str, Any]]:
    seen = set()
    out = []

    for ex in examples:
        key = (
            ex.get("query", "").strip().lower(),
            ex.get("gold_answer", "").strip().lower(),
            " ".join(ev["text"] for ev in ex.get("history", [])[:5]).strip().lower(),
        )

        if key in seen:
            continue

        seen.add(key)
        out.append(ex)

    # Reassign IDs after dedup.
    for i, ex in enumerate(out):
        ex["id"] = f"{dataset_name}_{i:06d}"

    return out


def write_jsonl(path: Path, rows: List[Dict[str, Any]]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)

    with path.open("w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")


def write_report(path: Path, report: Dict[str, Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)

    with path.open("w", encoding="utf-8") as f:
        json.dump(report, f, ensure_ascii=False, indent=2)


def convert_dataset(dataset_name: str, folder: Path, max_examples: Optional[int] = None) -> Tuple[List[Dict[str, Any]], Dict[str, Any]]:
    print("\n" + "=" * 100)
    print(f"Converting {dataset_name}")
    print("=" * 100)
    print(f"Folder: {folder}")

    raw_records = load_dataset_records(folder)

    print(f"Raw detected records: {len(raw_records)}")

    examples = []
    skipped = 0

    for i, rec in enumerate(raw_records):
        ex = convert_record_to_timebound(rec, dataset_name, i)
        if ex is None:
            skipped += 1
            continue
        examples.append(ex)

    examples = deduplicate_examples(examples, dataset_name)

    if max_examples is not None and len(examples) > max_examples:
        random.shuffle(examples)
        examples = examples[:max_examples]
        for i, ex in enumerate(examples):
            ex["id"] = f"{dataset_name}_{i:06d}"

    report = {
        "dataset_name": dataset_name,
        "folder": str(folder),
        "raw_detected_records": len(raw_records),
        "converted_examples": len(examples),
        "skipped_records": skipped,
        "max_examples": max_examples,
        "history_turns_min": min([len(x["history"]) for x in examples], default=0),
        "history_turns_max": max([len(x["history"]) for x in examples], default=0),
        "history_turns_avg": (
            sum(len(x["history"]) for x in examples) / len(examples)
            if examples else 0
        ),
        "heuristic_evidence_count": sum(
            1 for x in examples if x.get("metadata", {}).get("evidence_is_heuristic")
        ),
        "task_type_counts": {},
        "difficulty_counts": {},
        "sample_ids": [x["id"] for x in examples[:5]],
    }

    for ex in examples:
        report["task_type_counts"][ex["task_type"]] = report["task_type_counts"].get(ex["task_type"], 0) + 1
        report["difficulty_counts"][ex["difficulty"]] = report["difficulty_counts"].get(ex["difficulty"], 0) + 1

    return examples, report


# ============================================================
# Inspection helper
# ============================================================

def inspect_folders() -> None:
    print("=" * 100)
    print("Inspect dataset folders")
    print("=" * 100)
    print(f"Root: {ROOT}")

    for dataset_name, aliases in DATASET_ALIASES.items():
        folder = find_dataset_folder(ROOT, aliases)
        print("\n" + "-" * 100)
        print(dataset_name)

        if folder is None:
            print("  NOT FOUND")
            continue

        print(f"  folder: {folder}")
        files = list_supported_files(folder)
        print(f"  supported files: {len(files)}")

        for p in files[:30]:
            try:
                print(f"    {p.relative_to(folder)}  ({p.stat().st_size} bytes)")
            except Exception:
                print(f"    {p}")


def main():
    import argparse

    parser = argparse.ArgumentParser()

    parser.add_argument("--root", type=str, default=str(ROOT))
    parser.add_argument("--output_dir", type=str, default=str(OUTPUT_DIR))

    parser.add_argument(
        "--datasets",
        type=str,
        default="tempreason,complextr,tcp",
        help="Comma-separated: tempreason,complextr,tcp",
    )

    parser.add_argument(
        "--max_examples",
        type=int,
        default=None,
        help="Optional maximum examples per dataset.",
    )

    parser.add_argument(
        "--inspect_only",
        action="store_true",
        help="Only inspect folders and files.",
    )

    args, unknown = parser.parse_known_args()

    if unknown:
        print("Ignored unknown arguments:", unknown)

    root = Path(args.root)
    output_dir = Path(args.output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    if args.inspect_only:
        print("=" * 100)
        print("Inspect dataset folders")
        print("=" * 100)
        print(f"Root: {root}")

        for dataset_name, aliases in DATASET_ALIASES.items():
            folder = find_dataset_folder(root, aliases)
            print("\n" + "-" * 100)
            print(dataset_name)

            if folder is None:
                print("  NOT FOUND")
                continue

            print(f"  folder: {folder}")
            files = list_supported_files(folder)
            print(f"  supported files: {len(files)}")

            for p in files[:30]:
                try:
                    print(f"    {p.relative_to(folder)}  ({p.stat().st_size} bytes)")
                except Exception:
                    print(f"    {p}")

        return

    dataset_names = [x.strip().lower() for x in args.datasets.split(",") if x.strip()]

    all_reports = {}

    for dataset_name in dataset_names:
        if dataset_name not in DATASET_ALIASES:
            print(f"Unknown dataset key: {dataset_name}")
            continue

        folder = find_dataset_folder(root, DATASET_ALIASES[dataset_name])

        if folder is None:
            print(f"\nDataset folder not found for: {dataset_name}")
            all_reports[dataset_name] = {
                "dataset_name": dataset_name,
                "error": "folder_not_found",
                "aliases": DATASET_ALIASES[dataset_name],
            }
            continue

        examples, report = convert_dataset(
            dataset_name=dataset_name,
            folder=folder,
            max_examples=args.max_examples,
        )

        out_path = output_dir / f"{dataset_name}_timebound.jsonl"
        report_path = output_dir / f"{dataset_name}_report.json"

        write_jsonl(out_path, examples)
        write_report(report_path, report)

        all_reports[dataset_name] = report

        print("\nSaved:")
        print(f"  {out_path}")
        print(f"  {report_path}")
        print("\nReport:")
        print(json.dumps(report, ensure_ascii=False, indent=2))

    summary_path = output_dir / "conversion_summary.json"
    write_report(summary_path, all_reports)

    print("\n" + "=" * 100)
    print("DONE")
    print("=" * 100)
    print(f"Summary: {summary_path}")

if __name__ == "__main__":
    main()

Ignored unknown arguments: ['-f', 'C:\\Users\\ivan\\AppData\\Roaming\\jupyter\\runtime\\kernel-db0ca724-1d73-4239-812e-7e916c9f2a80.json']

Converting tempreason
Folder: D:\Users\TimeBound\TempReason
Found 10 supported files in D:\Users\TimeBound\TempReason
Raw detected records: 0

Saved:
  D:\Users\TimeBound\converted_external\tempreason_timebound.jsonl
  D:\Users\TimeBound\converted_external\tempreason_report.json

Report:
{
  "dataset_name": "tempreason",
  "folder": "D:\\Users\\TimeBound\\TempReason",
  "raw_detected_records": 0,
  "converted_examples": 0,
  "skipped_records": 0,
  "max_examples": null,
  "history_turns_min": 0,
  "history_turns_max": 0,
  "history_turns_avg": 0,
  "heuristic_evidence_count": 0,
  "task_type_counts": {},
  "difficulty_counts": {},
  "sample_ids": []
}

Converting complextr
Folder: D:\Users\TimeBound\complex-tr
Found 10 supported files in D:\Users\TimeBound\complex-tr
  test_gold_processed_gcs.json -> 329 records


In [ ]:
from pathlib import Path
import json
import csv
from collections import Counter, defaultdict

FOLDERS = [
    Path(r"D:\Users\TimeBound\TempReason"),
    Path(r"D:\Users\TimeBound\complex-tr"),
    Path(r"D:\Users\TimeBound\TCP"),
]

SUPPORTED_EXTS = {".json", ".jsonl", ".ndjson", ".csv", ".tsv", ".txt", ".parquet", ".arrow"}

MAX_FILES_TO_PRINT = 200
MAX_SAMPLE_CHARS = 2500


def human_size(n: int) -> str:
    units = ["B", "KB", "MB", "GB"]
    x = float(n)
    for u in units:
        if x < 1024:
            return f"{x:.1f} {u}"
        x /= 1024
    return f"{x:.1f} TB"


def safe_read_text(path: Path, max_chars: int = MAX_SAMPLE_CHARS) -> str:
    try:
        with path.open("r", encoding="utf-8", errors="ignore") as f:
            return f.read(max_chars)
    except Exception as e:
        return f"<cannot read as text: {e}>"


def load_json_preview(path: Path):
    try:
        text = path.read_text(encoding="utf-8", errors="ignore").strip()
        if not text:
            return "empty", None

        obj = json.loads(text)

        if isinstance(obj, list):
            first = obj[0] if obj else None
            return f"json list, len={len(obj)}", first

        if isinstance(obj, dict):
            return f"json dict, keys={list(obj.keys())[:30]}", obj

        return f"json {type(obj).__name__}", obj

    except Exception as e:
        return f"json parse failed: {e}", None


def load_jsonl_preview(path: Path, max_lines: int = 5):
    rows = []
    bad = 0
    total_checked = 0

    try:
        with path.open("r", encoding="utf-8", errors="ignore") as f:
            for i, line in enumerate(f):
                if i >= 2000:
                    break

                line = line.strip()
                if not line:
                    continue

                total_checked += 1

                try:
                    obj = json.loads(line)
                    if len(rows) < max_lines:
                        rows.append(obj)
                except Exception:
                    bad += 1

        return f"jsonl checked={total_checked}, parsed_sample={len(rows)}, bad={bad}", rows

    except Exception as e:
        return f"jsonl read failed: {e}", []


def load_csv_preview(path: Path, delimiter=",", max_rows: int = 5):
    try:
        with path.open("r", encoding="utf-8", errors="ignore", newline="") as f:
            reader = csv.DictReader(f, delimiter=delimiter)
            fields = reader.fieldnames
            rows = []
            for i, row in enumerate(reader):
                if i >= max_rows:
                    break
                rows.append(dict(row))

        return f"csv fields={fields}", rows

    except Exception as e:
        return f"csv read failed: {e}", []


def flatten_keys(obj, prefix="", max_depth=3):
    keys = []

    if max_depth <= 0:
        return keys

    if isinstance(obj, dict):
        for k, v in obj.items():
            key = f"{prefix}.{k}" if prefix else str(k)
            keys.append(key)
            keys.extend(flatten_keys(v, key, max_depth - 1))

    elif isinstance(obj, list) and obj:
        keys.extend(flatten_keys(obj[0], prefix + "[]", max_depth - 1))

    return keys


def summarize_sample(sample):
    if sample is None:
        return

    print("    SAMPLE TYPE:", type(sample).__name__)

    if isinstance(sample, dict):
        print("    TOP KEYS:", list(sample.keys())[:50])
        flat = flatten_keys(sample, max_depth=3)
        if flat:
            print("    FLAT KEYS:")
            for k in flat[:80]:
                print("      -", k)

        print("    SAMPLE VALUES:")
        for k, v in list(sample.items())[:15]:
            s = str(v)
            s = s.replace("\n", " ")
            if len(s) > 300:
                s = s[:300] + "..."
            print(f"      {k}: {s}")

    elif isinstance(sample, list):
        print("    LIST LEN:", len(sample))
        if sample:
            summarize_sample(sample[0])

    else:
        s = str(sample).replace("\n", " ")
        if len(s) > 800:
            s = s[:800] + "..."
        print("    SAMPLE:", s)


def inspect_file(path: Path, root: Path):
    rel = path.relative_to(root)
    size = path.stat().st_size
    ext = path.suffix.lower()

    print("\n" + "-" * 120)
    print(f"FILE: {rel}")
    print(f"SIZE: {human_size(size)}")
    print(f"EXT:  {ext}")

    if ext == ".json":
        info, sample = load_json_preview(path)
        print("    INFO:", info)

        # If top-level dict contains split lists, print those.
        if isinstance(sample, dict):
            for split_key in ["train", "validation", "valid", "dev", "test"]:
                if split_key in sample:
                    v = sample[split_key]
                    if isinstance(v, list):
                        print(f"    SPLIT {split_key}: list len={len(v)}")
                        if v:
                            print(f"    SPLIT {split_key} FIRST:")
                            summarize_sample(v[0])
                            return

        summarize_sample(sample)

    elif ext in {".jsonl", ".ndjson"}:
        info, rows = load_jsonl_preview(path)
        print("    INFO:", info)
        if rows:
            summarize_sample(rows[0])
        else:
            print("    RAW PREVIEW:")
            print(safe_read_text(path)[:1000])

    elif ext == ".csv":
        info, rows = load_csv_preview(path, delimiter=",")
        print("    INFO:", info)
        if rows:
            summarize_sample(rows[0])
        else:
            print("    RAW PREVIEW:")
            print(safe_read_text(path)[:1000])

    elif ext == ".tsv":
        info, rows = load_csv_preview(path, delimiter="\t")
        print("    INFO:", info)
        if rows:
            summarize_sample(rows[0])
        else:
            print("    RAW PREVIEW:")
            print(safe_read_text(path)[:1000])

    elif ext in {".txt"}:
        print("    RAW PREVIEW:")
        print(safe_read_text(path)[:1500])

    elif ext in {".parquet", ".arrow"}:
        print("    BINARY TABLE FILE.")
        print("    NOTE: not reading it here to avoid pyarrow/sklearn DLL issues.")
        print("    If this is the only real dataset file, we need a separate environment or pandas/pyarrow fix.")

    else:
        print("    RAW PREVIEW:")
        print(safe_read_text(path)[:1000])


def inspect_folder(folder: Path):
    print("\n" + "=" * 120)
    print("FOLDER:", folder)
    print("=" * 120)

    if not folder.exists():
        print("DOES NOT EXIST")
        return

    print("\nTOP-LEVEL CONTENT:")
    try:
        for p in sorted(folder.iterdir(), key=lambda x: (not x.is_dir(), x.name.lower())):
            typ = "DIR " if p.is_dir() else "FILE"
            size = "" if p.is_dir() else human_size(p.stat().st_size)
            print(f"  {typ:4} {p.name} {size}")
    except Exception as e:
        print("Cannot list top-level:", e)

    print("\nSCANNING SUPPORTED FILES...")
    files = []
    ext_counter = Counter()
    size_by_ext = defaultdict(int)

    try:
        for p in folder.rglob("*"):
            if not p.is_file():
                continue

            ext = p.suffix.lower()
            ext_counter[ext] += 1
            try:
                size_by_ext[ext] += p.stat().st_size
            except Exception:
                pass

            if ext in SUPPORTED_EXTS:
                files.append(p)

    except Exception as e:
        print("rglob failed:", e)
        return

    print("\nEXTENSION SUMMARY:")
    for ext, cnt in ext_counter.most_common(30):
        print(f"  {ext or '<no ext>':12} {cnt:6d} files   {human_size(size_by_ext[ext])}")

    print(f"\nSUPPORTED FILES FOUND: {len(files)}")

    files = sorted(files, key=lambda p: p.stat().st_size if p.exists() else 0, reverse=True)

    print("\nSUPPORTED FILES BY SIZE:")
    for p in files[:MAX_FILES_TO_PRINT]:
        try:
            print(f"  {p.relative_to(folder)}   {human_size(p.stat().st_size)}")
        except Exception:
            print(" ", p)

    # Inspect likely real data files first.
    likely = []
    metadata_keywords = [
        "readme",
        "license",
        "dataset_infos",
        "state",
        "cache",
        "lock",
        "metadata",
        "config",
        "schema",
    ]

    for p in files:
        name = p.name.lower()
        rel = str(p.relative_to(folder)).lower()

        if any(k in name for k in metadata_keywords):
            continue

        if p.suffix.lower() in {".json", ".jsonl", ".ndjson", ".csv", ".tsv"}:
            likely.append(p)

    if not likely:
        likely = [p for p in files if p.suffix.lower() in {".json", ".jsonl", ".ndjson", ".csv", ".tsv"}]

    print("\nLIKELY DATA FILES TO INSPECT:")
    for p in likely[:20]:
        print(f"  {p.relative_to(folder)}   {human_size(p.stat().st_size)}")

    print("\nDETAILED PREVIEW OF LIKELY FILES:")
    for p in likely[:10]:
        inspect_file(p, folder)


def main():
    for folder in FOLDERS:
        inspect_folder(folder)


if __name__ == "__main__":
    main()


FOLDER: D:\Users\TimeBound\TempReason

TOP-LEVEL CONTENT:
  DIR  .cache 
  FILE .gitattributes 2.6 KB
  FILE README.md 329.0 B
  FILE test_l1.json 641.0 KB
  FILE test_l1_future.json 319.4 KB
  FILE test_l2.json 64.3 MB
  FILE test_l3.json 59.3 MB
  FILE train_l1.json 63.3 MB
  FILE train_l2.json 157.9 MB
  FILE train_l3.json 140.2 MB
  FILE val_l1.json 640.9 KB
  FILE val_l2.json 55.9 MB
  FILE val_l3.json 51.1 MB

SCANNING SUPPORTED FILES...

EXTENSION SUMMARY:
  .metadata        12 files   1.4 KB
  .json            10 files   593.6 MB
  <no ext>          2 files   2.6 KB
  .md               1 files   329.0 B

SUPPORTED FILES FOUND: 10

SUPPORTED FILES BY SIZE:
  train_l2.json   157.9 MB
  train_l3.json   140.2 MB
  test_l2.json   64.3 MB
  train_l1.json   63.3 MB
  test_l3.json   59.3 MB
  val_l2.json   55.9 MB
  val_l3.json   51.1 MB
  test_l1.json   641.0 KB
  val_l1.json   640.9 KB
  test_l1_future.json   319.4 KB

LIKELY DATA FILES TO INSPECT:
  train_l2.json   157.9 MB
  train

In [1]:
from pathlib import Path
import json

files = [
    Path(r"D:\Users\TimeBound\TempReason\test_l1.json"),
    Path(r"D:\Users\TimeBound\TempReason\test_l1_future.json"),
    Path(r"D:\Users\TimeBound\complex-tr\test_gold.json"),
]

for path in files:
    print("\n" + "=" * 100)
    print(path)
    with path.open("r", encoding="utf-8", errors="ignore") as f:
        for i, line in enumerate(f):
            if i >= 3:
                break
            line = line.strip()
            print(f"\nLINE {i}:")
            print(line[:1000])
            try:
                obj = json.loads(line)
                print("KEYS:", list(obj.keys()))
            except Exception as e:
                print("JSON parse error:", e)


D:\Users\TimeBound\TempReason\test_l1.json

LINE 0:
{"question": "What is the time 6 year and 4 month after Nov, 1185", "date": "November 18, 1185", "text_answers": {"text": ["Mar, 1192"]}, "id": "0", "context": ""}
KEYS: ['question', 'date', 'text_answers', 'id', 'context']

LINE 1:
{"question": "What is the time 4 year and 12 month after May, 1742", "date": "May 15, 1742", "text_answers": {"text": ["May, 1747"]}, "id": "1", "context": ""}
KEYS: ['question', 'date', 'text_answers', 'id', 'context']

LINE 2:
{"question": "What is the time 7 year and 5 month after Oct, 1241", "date": "October 21, 1241", "text_answers": {"text": ["Mar, 1249"]}, "id": "2", "context": ""}
KEYS: ['question', 'date', 'text_answers', 'id', 'context']

D:\Users\TimeBound\TempReason\test_l1_future.json

LINE 0:
{"question": "What is the time 3 year and 3 month after Sep, 2038", "date": "September 05, 2038", "text_answers": {"text": ["Dec, 2041"]}, "id": "0", "context": ""}
KEYS: ['question', 'date', 'text_answ